In [ ]:
# Example 1: Basic Progress Bar Usage
from tqdm import tqdm
import time
import random

# Basic progress bar for simple loops
print("Basic progress bar demo:")
for i in tqdm(range(10), desc="Processing"):
    time.sleep(0.1)  # Simulate work

# Progress bar with custom description and unit
print("\nCustom progress bar with units:")
data_size = 50
for item in tqdm(range(data_size), desc="Downloading", unit="MB"):
    time.sleep(0.05)  # Simulate download

# Manual progress bar updates
print("\nManual progress updates:")
pbar = tqdm(total=100, desc="Manual Progress")
for i in range(10):
    # Simulate variable work amounts
    work_done = random.randint(5, 15)
    pbar.update(work_done)
    time.sleep(0.1)
pbar.close()

print("Basic progress bar usage complete")

In [ ]:
# Example 2: Advanced Progress Bar Configuration and Styling
from tqdm import tqdm
import time
import sys

# Custom progress bar formatting
print("Custom formatted progress bar:")
custom_format = "{l_bar}{bar:30}{r_bar}{bar:-30b}"
for i in tqdm(range(20), bar_format=custom_format, desc="Custom Format"):
    time.sleep(0.05)

# Progress bar with detailed statistics
print("\nProgress bar with detailed statistics:")
for i in tqdm(range(30), desc="Processing", unit="items", 
              miniters=1, ascii=True, dynamic_ncols=True):
    time.sleep(0.03)

# Nested progress bars
print("\nNested progress bars:")
outer_tasks = 3
inner_tasks = 10

outer_pbar = tqdm(range(outer_tasks), desc="Outer Loop", position=0)
for i in outer_pbar:
    outer_pbar.set_description(f"Outer Loop {i+1}/{outer_tasks}")
    
    inner_pbar = tqdm(range(inner_tasks), desc="Inner Loop", position=1, leave=False)
    for j in inner_pbar:
        inner_pbar.set_description(f"Inner {j+1}/{inner_tasks}")
        time.sleep(0.02)
    inner_pbar.close()

outer_pbar.close()
print("Advanced progress bar configuration complete")

In [ ]:
# Example 3: File Processing and Data Analysis with Progress Tracking
from tqdm import tqdm
import time
import random
import tempfile
import os

def process_large_dataset():
    """Simulate processing a large dataset with progress tracking"""
    
    # Simulate reading a large file
    print("Simulating large file processing:")
    file_size = 1000  # Simulate 1000 lines
    
    processed_data = []
    with tqdm(total=file_size, desc="Reading file", unit="lines") as pbar:
        for line_num in range(file_size):
            # Simulate variable processing time per line
            processing_time = random.uniform(0.001, 0.005)
            time.sleep(processing_time)
            
            # Simulate data processing
            processed_data.append(f"processed_line_{line_num}")
            
            # Update progress bar with current line info
            if line_num % 100 == 0:
                pbar.set_postfix(current_line=line_num, processed=len(processed_data))
            pbar.update(1)
    
    return processed_data

def analyze_data(data):
    """Analyze processed data with progress tracking"""
    print("\nAnalyzing processed data:")
    
    analysis_results = {}
    batch_size = 50
    
    # Process data in batches
    for i in tqdm(range(0, len(data), batch_size), desc="Analysis", unit="batch"):
        batch = data[i:i+batch_size]
        
        # Simulate analysis
        time.sleep(0.02)
        analysis_results[f"batch_{i//batch_size}"] = {
            "size": len(batch),
            "avg_length": sum(len(item) for item in batch) / len(batch)
        }
    
    return analysis_results

# Execute the data processing pipeline
processed_data = process_large_dataset()
analysis_results = analyze_data(processed_data)

print(f"\nProcessing complete:")
print(f"Processed {len(processed_data)} items")
print(f"Generated {len(analysis_results)} analysis batches")
print("File processing and data analysis with progress tracking complete")

In [ ]:
# Example 4: Parallel Processing with Multiple Progress Bars
from tqdm import tqdm
import time
import threading
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

def worker_task(task_id, num_items, pbar_position):
    """Simulate a worker task with its own progress bar"""
    pbar = tqdm(range(num_items), desc=f"Worker {task_id}", position=pbar_position, leave=True)
    
    results = []
    for i in pbar:
        # Simulate variable work time
        work_time = random.uniform(0.01, 0.05)
        time.sleep(work_time)
        
        result = f"task_{task_id}_item_{i}"
        results.append(result)
        
        # Update progress bar with status info
        pbar.set_postfix(processed=len(results), rate=f"{1/work_time:.1f}/s")
    
    pbar.close()
    return results

def parallel_processing_demo():
    """Demonstrate parallel processing with multiple progress bars"""
    print("Parallel processing with multiple progress bars:")
    
    num_workers = 3
    items_per_worker = 20
    
    # Create a main progress bar for overall completion
    main_pbar = tqdm(total=num_workers, desc="Overall Progress", position=0)
    
    # Run workers in parallel
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        # Submit tasks
        futures = []
        for worker_id in range(num_workers):
            future = executor.submit(worker_task, worker_id, items_per_worker, worker_id + 1)
            futures.append(future)
        
        # Collect results as they complete
        all_results = []
        for future in as_completed(futures):
            result = future.result()
            all_results.extend(result)
            main_pbar.update(1)
    
    main_pbar.close()
    return all_results

# Execute parallel processing demo
results = parallel_processing_demo()
print(f"\nParallel processing completed: {len(results)} total items processed")
print("Parallel processing with multiple progress bars complete")

In [ ]:
# Example 5: Production Monitoring and Performance Analysis
from tqdm import tqdm
import time
import random
import statistics
from datetime import datetime, timedelta
import sys

class SimulatedProcessingError(Exception):
    """The one failure this demo injects on purpose."""

class ProductionMonitor:
    """Production-ready monitoring with tqdm progress bars"""
    
    def __init__(self):
        self.metrics = {
            'processing_times': [],
            'throughput': [],
            'error_count': 0,
            'total_processed': 0
        }
        self.start_time = None
    
    def process_batch(self, batch_data, pbar):
        """Process a batch of data with performance monitoring"""
        batch_start = time.time()
        processed_items = []
        
        for item in batch_data:
            item_start = time.time()
            
            try:
                # Simulate processing with occasional errors: deterministically every 20th ATTEMPT
                # (a random 5% rate made the error count differ run to run; keyed on the attempt,
                # not the item, because a failed item is retried and must not fail forever)
                if self.metrics['total_processed'] % 20 == 19:
                    raise SimulatedProcessingError(f"Simulated processing failure on item {item}")
                
                # Simulate variable processing time
                processing_time = random.uniform(0.01, 0.03)
                time.sleep(processing_time)
                
                processed_items.append(f"processed_{item}")
                self.metrics['processing_times'].append(time.time() - item_start)
                
            except SimulatedProcessingError:
                self.metrics['error_count'] += 1
            
            self.metrics['total_processed'] += 1
            
            # Update progress bar with real-time metrics
            if self.metrics['processing_times']:
                avg_time = statistics.mean(self.metrics['processing_times'][-100:])  # Last 100 items
                throughput = 1 / avg_time if avg_time > 0 else 0
                
                pbar.set_postfix(
                    processed=self.metrics['total_processed'],
                    errors=self.metrics['error_count'],
                    avg_time=f"{avg_time:.3f}s",
                    throughput=f"{throughput:.1f}/s"
                )
        
        batch_time = time.time() - batch_start
        batch_throughput = len(processed_items) / batch_time if batch_time > 0 else 0
        self.metrics['throughput'].append(batch_throughput)
        
        return processed_items
    
    def run_production_job(self, total_items, batch_size=50):
        """Run a production job with comprehensive monitoring"""
        self.start_time = time.time()
        print(f"Starting production job at {datetime.now().strftime('%H:%M:%S')}")
        
        # Create main progress bar
        with tqdm(total=total_items, desc="Production Job", unit="items",
                  bar_format='{l_bar}{bar:30}{r_bar}{bar:-30b}') as pbar:
            
            processed_count = 0
            
            while processed_count < total_items:
                # Create batch
                current_batch_size = min(batch_size, total_items - processed_count)
                batch_data = list(range(processed_count, processed_count + current_batch_size))
                
                # Process batch
                processed_batch = self.process_batch(batch_data, pbar)
                processed_count += len(processed_batch)
                
                # Update main progress bar
                pbar.update(len(processed_batch))
        
        total_time = time.time() - self.start_time
        return self.generate_performance_report(total_time)
    
    def generate_performance_report(self, total_time):
        """Generate comprehensive performance report"""
        report = []
        report.append("Production Job Performance Report")
        report.append("=" * 40)
        report.append(f"Total execution time: {total_time:.2f} seconds")
        report.append(f"Total items processed: {self.metrics['total_processed']}")
        report.append(f"Total errors: {self.metrics['error_count']}")
        
        if self.metrics['processing_times']:
            avg_processing_time = statistics.mean(self.metrics['processing_times'])
            report.append(f"Average processing time: {avg_processing_time:.3f}s")
            report.append(f"Processing time std dev: {statistics.stdev(self.metrics['processing_times']):.3f}s")
        
        if self.metrics['throughput']:
            avg_throughput = statistics.mean(self.metrics['throughput'])
            report.append(f"Average throughput: {avg_throughput:.1f} items/s")
        
        overall_throughput = self.metrics['total_processed'] / total_time
        report.append(f"Overall throughput: {overall_throughput:.1f} items/s")
        
        error_rate = (self.metrics['error_count'] / self.metrics['total_processed']) * 100
        report.append(f"Error rate: {error_rate:.2f}%")
        
        return "\n".join(report)

# Demonstrate production monitoring
monitor = ProductionMonitor()
performance_report = monitor.run_production_job(total_items=200, batch_size=25)

print("\n" + performance_report)
# Every failed item is retried in the next batch, so exactly `total_items` succeed and every error is counted.
assert monitor.metrics['total_processed'] - monitor.metrics['error_count'] == 200, monitor.metrics
assert monitor.metrics['error_count'] == 10, monitor.metrics['error_count']  # attempts 19, 39, ..., 199 of 210
assert len(monitor.metrics['processing_times']) == 200, len(monitor.metrics['processing_times'])
print("\nProduction monitoring and performance analysis complete")